In [1]:
import torch
import torch.nn.functional as F
import numpy as np
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer
from tqdm import tqdm
import json
import textwrap
import sys
import os

In [16]:
RUN = 0

In [17]:
if RUN ==0 :
  from datasets import load_dataset

  # dataset = load_dataset("databricks/databricks-dolly-15k")
  dataset = load_dataset(cfg.SFT_DATASET_NAME)
  sft_data = dataset["train"][0:cfg.SFT_DATA_SIZE]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [4]:
from training.sft_data_utils import create_instruction, tokenize_sft_dataset, save_sft_tokens, load_sft_tokens, get_sft_batch

In [5]:
# ── Environment setup ─────────────────────────────────────────────────────
# Run this cell first every Colab session (or once on a local machine).
# It mounts Drive (Colab only) and adds the repo root to sys.path so all
# project imports work correctly.
import os, sys

if os.path.exists("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Add the repo root to the module search path
sys.path.insert(0, os.path.abspath(".."))   # works locally from notebooks/
                                             # adjust to REPO_ROOT on Colab

import config as cfg
cfg.make_dirs()
cfg.print_config()

Mounted at /content/drive


In [6]:
from model import GPT, GPTConfig, GPTWithKVCache
config = GPTConfig()
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("device:", device)

In [7]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer(
    cfg.TOKENIZER_VOCAB,
    cfg.TOKENIZER_MERGES
)

eos_id = tokenizer.token_to_id("</s>")

In [18]:
import pickle
SFT_SAMPLE_PATH = cfg.SFT_TOKENS
if RUN ==0:
  samples = []

  for example in tqdm(sft_data["text"]):

      story = example.replace("\n"," ")
      text = create_instruction(story)

      tokens = tokenizer.encode(text).ids
      tokens.append(eos_id)

      samples.append(tokens)

  print("Total samples:", len(samples))

  with open(SFT_SAMPLE_PATH, "wb") as f:
      pickle.dump(samples, f)

  print("Saved tokenized samples to Google Drive")

100%|██████████| 100000/100000 [01:13<00:00, 1363.71it/s]


Total samples: 100000
Saved tokenized samples to Google Drive


In [19]:
with open(SFT_SAMPLE_PATH, "rb") as f:
    samples = pickle.load(f)

print("Total samples:", len(samples))

Total samples: 100000


In [20]:
print("Example sample length:", len(samples[0]))
print("Max token:", max(max(s) for s in samples[:1000]))
print("Tokenizer vocab:", tokenizer.get_vocab_size())

Example sample length: 177
Max token: 8183
Tokenizer vocab: 8192


cuda


In [23]:
if device == "cpu":
  torch.set_num_threads(2)

In [24]:

if device == "cpu":
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT, map_location="cpu")
else:
  ckpt = torch.load(cfg.PRETRAIN_FINAL_CKPT)

model.load_state_dict(ckpt)

# model.eval()
print("Model loaded successfully")

Model loaded successfully


In [31]:
response_tokens = tokenizer.encode(" Response:").ids
response_tokens = torch.tensor(response_tokens, dtype=torch.long).to(device).tolist()
response_len = len(response_tokens)

block_size = config.block_size
batch_size = 16

import torch
import numpy as np

block_size = config.block_size

def get_batch():

    idx = torch.randint(len(samples), (batch_size,))
    x_batch = []
    y_batch = []

    for i in idx:
        tokens = samples[i]
        # truncate
        if len(tokens) > block_size:
            tokens = tokens[:block_size]
        # pad
        else:
            tokens = tokens + [1] * (block_size - len(tokens))
        x = torch.tensor(tokens[:-1])
        y = torch.tensor(tokens[1:])

        # ----- masking -----

        response_start = None

        for j in range(len(tokens) - len(response_tokens)):

            if tokens[j:j+len(response_tokens)] == response_tokens:

                response_start = j + len(response_tokens)
                break
        if response_start is not None:

            y[:response_start-1] = -100

        x_batch.append(x)
        y_batch.append(y)

    x = torch.stack(x_batch).to(device)
    y = torch.stack(y_batch).to(device)

    return x, y

In [38]:
x, y = get_batch()

decoded = tokenizer.decode(x[0].tolist())
print(textwrap.fill(decoded, 80))

print("Masked tokens:", (y[0] == -100).sum())

Instruction: Write a story for kids. Response: Once there was a little frog who
wanted to see the world. He leapt from his home and began hopping from place to
place. Finally he came to the edge of the ocean. He had never seen the ocean
before, and was excited to explore it.   The frog hopped into the water and soon
found himself surrounded by the salty water and bubbles. He felt so free and
happy. He stayed there for a few minutes, but then he started feeling
uncomfortable. Suddenly the ocean started rising, growing bigger and bigger!
He realized he was in danger, and he needed to get out of the ocean quickly. He
jumped and hopped as far as he could, but the waves kept getting higher and
higher. He couldn't keep up with them, and the ocean soon covered him, taking
him away from his home.   The moral of the story is to listen to your instincts.
Even though the frog wanted to explore, he should have been more careful and not
stayed in the ocean when he started to feel uncomfortable.</s>

In [41]:
y_clean = torch.where(y[0] == -100, 1, y[0])
y_text = tokenizer.decode(y_clean.tolist())
print(textwrap.fill(y_text, 80))

<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
<pad><pad> Once there was a little frog who wanted to see the world. He leapt
from his home and began hopping from place to place. Finally he came to the edge
of the ocean. He had never seen the ocean before, and was excited to explore it.
The frog hopped into the water and soon found himself surrounded by the salty
water and bubbles. He felt so free and happy. He stayed there for a few minutes,
but then he started feeling uncomfortable. Suddenly the ocean started rising,
growing bigger and bigger!   He realized he was in danger, and he needed to get
out of the ocean quickly. He jumped and hopped as far as he could, but the waves
kept getting higher and higher. He couldn't keep up with them, and the ocean
soon covered him, taking him away from his home.   The moral of the story is to
listen to your instincts. Even though the frog wanted to explore, he should have
been more careful and not stayed in the oce

In [27]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.SFT_LR,
    weight_decay=cfg.SFT_WEIGHT_DECAY
)

In [28]:
ckpt_path = cfg.SFT_CKPT
start_step = 0
if os.path.exists(ckpt_path):
  if device == "cpu":
    ckpt = torch.load(ckpt_path, map_location="cpu")
  else:
    ckpt = torch.load(ckpt_path)

  model.load_state_dict(ckpt["model"])
  optimizer.load_state_dict(ckpt["optimizer"])

  start_step = ckpt["step"]

In [29]:
start_step

0

In [32]:
max_steps = cfg.SFT_MAX_STEPS
eval_interval = cfg.SFT_EVAL_INTERVAL
save_interval = cfg.SFT_SAVE_INTERVAL

for step in range(start_step, max_steps):

    x, y = get_batch()
    logits = model(x)

    loss = F.cross_entropy(
        logits.view(-1, config.vocab_size),
        y.view(-1),
        ignore_index=-100
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"step {step} | loss {loss.item():.4f}")

    if step % save_interval == 0:

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step
            },
            ckpt_path
        )

torch.save(
  {
      "model": model.state_dict()
  },
  cfg.SFT_FINAL_CKPT
)

step 0 | loss 5.7571
step 50 | loss 1.6733
step 100 | loss 1.4886
step 150 | loss 1.5782
step 200 | loss 1.4244
step 250 | loss 1.5627
step 300 | loss 1.2935
step 350 | loss 1.2442
step 400 | loss 1.3526
step 450 | loss 1.2901
step 500 | loss 1.3408
step 550 | loss 1.2286
step 600 | loss 1.3619
step 650 | loss 1.3371
step 700 | loss 1.2966
step 750 | loss 1.2143
step 800 | loss 1.4285
step 850 | loss 1.3022
step 900 | loss 1.4044
step 950 | loss 1.6193
step 1000 | loss 1.3702
step 1050 | loss 1.4054
step 1100 | loss 1.2028
step 1150 | loss 1.4162
step 1200 | loss 1.2621
step 1250 | loss 1.2103
step 1300 | loss 1.1656
step 1350 | loss 1.2333
step 1400 | loss 1.2831
step 1450 | loss 1.3194
step 1500 | loss 1.1714
step 1550 | loss 1.3919
step 1600 | loss 1.3874
step 1650 | loss 1.3119
step 1700 | loss 1.3388
step 1750 | loss 1.4354
step 1800 | loss 1.2377
step 1850 | loss 1.2441
step 1900 | loss 1.2875
step 1950 | loss 1.3933
step 2000 | loss 1.3788
step 2050 | loss 1.4659
step 2100 | los

In [33]:
from generation.sampler import generate, encode_prompt

In [45]:
%%time
prompts = [
    "Write a short children's story.",
    "Tell a bedtime story.",
    "Write a story for kids.",
    "Create a short fairy tale.",
    "Tell a story about friendship."
]
prompt = random.choice(prompts)

input_ids = tokenizer.encode(prompt).ids
x = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

output = generate(
    model,
    x,
    max_new_tokens=200,
    temperature=1,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.1
)
print("\n Output without KV cache")
tokens = output[0].tolist()
text = tokenizer.decode(tokens)
text = f"Instruction: {prompt} \n Response: {text}"
print(textwrap.fill(text, 80))



 Output without KV cache
Instruction: Tell a bedtime story.   Response: Tell a bedtime story.  Sam does
not like this. He wants to play more. He does not want to share his toys with
anyone. But he likes the truck. He goes to Sam and says, "Please, can I have
some of your truck? Can I trade?"  Sam looks at Ben and shakes his head. He does
not know how to trade. He thinks hard about what the girl has. He wonders if he
can trade with him too. He decides to trade.  Sam takes some of his toy cars and
breaks them apart. He gives one car to Ben and says, "Here you go. I want to
trade for a trade."  Ben is angry. He is still sad. He sees Sam was still
unhappy. He says, "Hey, that is not fair. You cannot have two cars today. You
should say sorry."  Sam hears Ben and gets angry too. He stops making noises. He
says, "Fine. You are better than me. Now I am bigger and
CPU times: user 1.77 s, sys: 2.94 ms, total: 1.78 s
Wall time: 1.8 s
